# Tweet Narratives

Interactive retrieval of the tweets closest in meaning to a query, then the
webapp's write-and-grade loop on the result: a summary under either prompt set
(frames or narratives), checked by the three graders — word budget, ACUEval,
rubric (src/llm/pipeline, through OpenRouter).

In [1]:
# === Colab setup — automatically skipped when running locally ===
# Reads the project and its data from the shared PONS folder in Drive. 
# Require a shortcut to PONS folder from myDrive
# google.colab only exists on Colab, so a local type checker cannot resolve it.
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_DIR = ("/content/drive/MyDrive/PONS/EXPERIMENTS/TERM-CORRELATION"
                   "/Interactive_Component_Landscape")

    # 1. Mount Google Drive.
    from google.colab import drive  # ty: ignore[unresolved-import]
    drive.mount("/content/drive")

    assert os.path.isdir(PROJECT_DIR), (
        f"{PROJECT_DIR} not found — add a shortcut to the shared PONS folder "
        "in My Drive (Shared with me > PONS > Organise > Add shortcut).")

    # 2. Read the data from that folder, and make the src/ modules importable.
    os.environ["DATA_DIR"] = f"{PROJECT_DIR}/data"
    sys.path.insert(0, f"{PROJECT_DIR}/src")

    # 3. Install the packages Colab doesn't already ship with.
    %pip install -q -r "{PROJECT_DIR}/requirements-colab.txt" sentence-transformers openai

    # 4. ipywidgets interactivity on Colab needs the custom widget manager.
    from google.colab import output  # ty: ignore[unresolved-import]
    output.enable_custom_widget_manager()

In [ ]:
# Setup
import math
import sys
from typing import Any

import ipywidgets as widgets
from IPython.display import display, Markdown

sys.path.insert(0, ".")  

from rag.tweet_retrieval import bare_names, get_tweets_about
from llm.pipeline.budget import TOLERANCE, build_budget_feedback, word_count
from llm.pipeline.levels import LEVELS
from ica.load_data import load_embedding
from ica.decomposition import load_or_compute_ica_embeddings, get_components_with_strong_words
from ica.get_speakers import get_strong_speakers, get_extreme_speakers

# Any identifier OpenRouter knows is valid. The webapp's boxes default to
# openai/gpt-5.6-luna (writer, rubric) and z-ai/glm-5.3-flash (ACUEval).
MODEL = "openai/gpt-oss-20b:nitro" 

# The prompt set — frames or narratives — is the Prompts dropdown below; the
# agents are built per click from its choice, as the webapp builds them per
# call. Always at the word level: this page writes about one query, never a
# word list (src/llm/pipeline/levels.py).

In [3]:
# Data: embedding, ICA and retained components (same selection as the Component Landscape notebook)
emb = load_embedding()
ica_embedding = load_or_compute_ica_embeddings(emb.vectors)
selected_components = get_components_with_strong_words(ica_embedding, emb, strong_word_ratio_threshold=0.007)
print(f"{len(selected_components)} retained components: {selected_components}")

13 retained components: [6, 19, 26, 29, 46, 47, 69, 78, 81, 100, 102, 166, 201]


In [4]:
# Speaker selection, cached so widget interaction stays responsive
_speaker_cache = {}  # (source, component, k, phase) -> set of bare speaker names


def selected_speakers(source, component, k, phase):
    """The speakers to restrict retrieval to, or None for every speaker.

    Strong speakers are a global assignment (each speaker is strong for
    exactly one component), so the phase only restricts which tweets are
    searched. Extreme speakers are ranked within the phase when one is
    selected, as in the Component Landscape notebook.
    """
    if source == "all":
        return None
    key = (source, component, k if source == "extreme speakers" else None,
           phase if source == "extreme speakers" else None)
    if key not in _speaker_cache:
        if source == "strong speakers":
            tokens = get_strong_speakers(ica_embedding, emb, component)
        else:
            tokens = get_extreme_speakers(ica_embedding, emb, component, k=k, phase=phase)
        _speaker_cache[key] = bare_names(tokens)
    return _speaker_cache[key]

In [ ]:
# Retrieval, and the summary on demand
# What the summary button summarises: `render` fills it and only then enables
# the button, so by the time `on_summary` reads it both keys hold real values.
_last: dict[str, Any] = {"query": None, "tweets": None}

narrative_out = widgets.Output(layout=widgets.Layout(max_width="85ch"))


def render(query, n_tweets, phase, speakers, component, n_extreme, min_similarity):
    narrative_out.clear_output()
    w_summary.disabled = True
    query = query.strip()
    if not query:
        print("Type a query and press Enter.")
        return

    phase = None if phase == "all" else phase
    names = selected_speakers(speakers, component, n_extreme, phase)
    if names is not None and not names:
        print(f"Component {component} has no {speakers}.")
        return

    tweets = get_tweets_about(query, take_n=n_tweets,
                              phase=None if phase is None else int(phase),
                              speakers=names, min_similarity=min_similarity)
    if not len(tweets):
        print(f"No tweet within {min_similarity} similarity of the query "
              "— lower the floor or rephrase.")
        return

    _last.update(query=query, tweets=tweets["tweet"].tolist())
    w_summary.disabled = False
    scope = "" if names is None else f", from {len(names)} {speakers} of component {component}"
    print(f"{len(tweets)} tweets{scope}:")
    with pd_full_width():
        display(tweets)


def pd_full_width():
    import pandas as pd
    return pd.option_context("display.max_colwidth", None)


def on_summary(_):
    # The word level of the chosen prompt set, then the webapp's three graders
    # on the result: the word budget (arithmetic, no model), ACUEval's
    # fact-by-fact verification against the tweets, and the rubric.
    level = LEVELS[w_prompts.value].word
    with narrative_out:
        print("Asking the LLM…")
        summary = level.writer(model=MODEL).write(_last["query"], _last["tweets"])
        narrative_out.clear_output(wait=True)
        display(Markdown(summary))

        over = build_budget_feedback(summary, level.words, TOLERANCE, level.budget_feedback)
        print(f"Budget: {word_count(summary)} words of {level.words}"
              + (" — over" if over else ""))

        verified, acu_score = level.agent("acueval", model=MODEL).evaluate_summary(
            summary, _last["tweets"])
        supported = verified["supported"]
        score = "n/a" if math.isnan(acu_score) else f"{acu_score:.2f}"
        print(f"ACUEval: {score} — {int(supported.fillna(False).sum())} of "
              f"{len(supported)} units supported")
        with pd_full_width():
            display(verified)

        grades = level.agent("rubric", model=MODEL).evaluate_summary(
            _last["query"], summary, _last["tweets"])
        with pd_full_width():
            display(grades if grades is not None else "The grader did not return grades.")

In [ ]:
# Widgets and wiring.
_wide_label = {"description_width": "130px"}
w_query = widgets.Text(description="Query", placeholder="free text — press Enter",
                       continuous_update=False, style=_wide_label,
                       layout=widgets.Layout(width="400px"))
w_n_tweets = widgets.BoundedIntText(value=20, min=1, max=500, description="N tweets", style=_wide_label)
w_phase = widgets.Dropdown(options=["all", "1", "2", "3", "4"], value="all",
                           description="Tweets in phase", style=_wide_label)
# Cosine floor: a tweet is only returned if it is at least this close to the
# query. Raise it for stricter matches, lower it to see more.
w_min_similarity = widgets.FloatSlider(value=0.3, min=0.0, max=0.9, step=0.05,
                                       description="Min. similarity", readout_format=".2f",
                                       continuous_update=False, style=_wide_label)
w_speakers = widgets.RadioButtons(options=["all", "strong speakers", "extreme speakers"],
                                  description="Tweets from", style=_wide_label)
w_component = widgets.Dropdown(options=selected_components, description="Component",
                               disabled=True, style=_wide_label)
w_n_extreme = widgets.BoundedIntText(value=50, min=1, max=5000, description="N extreme speakers",
                                     disabled=True, style=_wide_label)


def _on_speakers(change):
    w_component.disabled = change["new"] == "all"
    w_n_extreme.disabled = change["new"] != "extreme speakers"

w_speakers.observe(_on_speakers, names="value")

# The webapp's tab bar as a dropdown, frames first and the default, as there
# (src/webapp/params.py PROMPT_LABELS and DEFAULT_PROMPT_SET).
w_prompts = widgets.Dropdown(options=[("Frames", "frames"), ("Narratives", "narratives")],
                             value="frames", description="Prompts",
                             layout=widgets.Layout(width="220px"))
w_summary = widgets.Button(description="Write & grade", button_style="primary",
                           disabled=True,
                           tooltip="Summarise the retrieved tweets under the chosen "
                                   "prompts, then grade the summary")
w_summary.on_click(on_summary)

_column_layout = widgets.Layout(margin="0 50px 10px 0")
ui = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Tweets</b>"), w_query, w_n_tweets, w_phase,
                  w_min_similarity],
                 layout=_column_layout),
    widgets.VBox([widgets.HTML("<b>Speakers</b>"), w_speakers, w_component, w_n_extreme],
                 layout=_column_layout),
], layout=widgets.Layout(flex_flow="row wrap"))

out = widgets.interactive_output(render, {
    "query": w_query, "n_tweets": w_n_tweets, "phase": w_phase,
    "speakers": w_speakers, "component": w_component, "n_extreme": w_n_extreme,
    "min_similarity": w_min_similarity,
})
display(ui, widgets.VBox([widgets.HBox([w_prompts, w_summary]), narrative_out]), out)